# Detektor anomalii w ruchu sieciowym

Plik `train.csv.gz` (w tym samym folderze co notatnik) zawiera kilka dni **normalnego** ruchu ośmiu sieci małych
firm – przepływy sieciowe z sondy Tstat, bez etykiet. Kolumna `network` mówi, z której sieci pochodzi przepływ. Zbuduj detektor, który nauczy się tego ruchu i będzie oceniał
nowe przepływy na dwa sposoby:

- `score(df)` – **ryzyko**: liczba dla każdego przepływu, im wyżej, tym bardziej podejrzany,
- `predict(df)` – **decyzja**: 1 = alarm, 0 = brak alarmu.

Efektem pracy jest plik **`model.pkl`** – wytrenowany detektor zapisany na końcu notatnika.

Kolumny: `network`, `ts` (s, UTC), `duration` (s), `src_ip`, `src_port`, `dst_ip`, `dst_port`, `proto` (tcp/udp),
`state` (SF, S0, REJ, RSTO, RSTR, OTH), `bytes_out`, `bytes_in`, `pkts_out`, `pkts_in`,
`sni` (nazwa serwera z TLS), `fqdn` (nazwa z DNS/HTTP, jeśli znana).


In [55]:
import os
import time
import numpy as np
import pandas as pd
import cloudpickle
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import pairwise_distances_argmin_min
from sklearn.preprocessing import StandardScaler

## Dane

In [56]:
train = pd.read_csv("train.csv.gz", keep_default_na=False)
print(f"{len(train):,} przepływów")
train.head()


1,664,769 przepływów


,network,ts,duration,src_ip,src_port,dst_ip,dst_port,proto,state,bytes_out,bytes_in,pkts_out,pkts_in,sni,fqdn
0,siec_03,1.741392e+09,0.0380,10.149.209.204,48126,10.215.9.214,445,tcp,SF,194,163,6,4,,
1,siec_03,1.741392e+09,0.0180,10.149.209.204,48128,10.215.9.214,445,tcp,SF,194,163,6,4,,
2,siec_03,1.741392e+09,0.0144,10.149.209.204,48130,10.215.9.214,445,tcp,SF,194,163,6,4,,
3,siec_03,1.741392e+09,0.0137,10.149.209.204,48132,10.215.9.214,445,tcp,SF,194,163,6,4,,
4,siec_07,1.741392e+09,0.0241,10.213.193.212,49296,10.244.46.120,445,tcp,SF,194,163,6,4,,


## Cechy

In [57]:
FAILED = {"S0", "REJ", "RSTO", "RSTR", "OTH"}


def flow_features(df):
    """Tanie cechy pojedynczego przepływu."""
    out = pd.DataFrame(index=df.index)
    for col in ("bytes_out", "bytes_in", "pkts_out", "duration"):
        out[f"log_{col}"] = np.log1p(pd.to_numeric(df[col], errors="coerce").clip(lower=0))
    out["out_in_ratio"] = out["log_bytes_out"] - np.log1p(
        pd.to_numeric(df.bytes_in, errors="coerce").clip(lower=0))
    out["failed"] = df.state.isin(FAILED).astype(np.float32)
    out["udp"] = (df.proto == "udp").astype(np.float32)
    out["dst_port_low"] = (pd.to_numeric(df.dst_port, errors="coerce") < 1024).astype(np.float32)
    hour = (pd.to_numeric(df.ts, errors="coerce") % 86400) / 3600
    out["hour_sin"] = np.sin(2 * np.pi * hour / 24)
    out["hour_cos"] = np.cos(2 * np.pi * hour / 24)
    return out.replace([np.inf, -np.inf], np.nan).fillna(0).astype(np.float32)


def window_features(df, window=60):
    """Agregaty per źródło i cel w minucie; kontekst liczony dla przekazanej ramki."""
    w = (pd.to_numeric(df.ts, errors="coerce").fillna(0) // window).astype(np.int64)
    tmp = df.assign(_w=w, _failed=df.state.isin(FAILED).astype(np.float32))
    g = tmp.groupby(["src_ip", "_w"], sort=False)
    src = pd.DataFrame({"src_n_flows": g.size(), "src_n_dst": g.dst_ip.nunique(),
                        "src_n_dport": g.dst_port.nunique(), "src_failed_ratio": g._failed.mean(),
                        "src_bytes_out": g.bytes_out.sum()})
    g = tmp.groupby(["dst_ip", "_w"], sort=False)
    dst = pd.DataFrame({"dst_n_flows": g.size(), "dst_n_src": g.src_ip.nunique(),
                        "dst_n_dport": g.dst_port.nunique(), "dst_failed_ratio": g._failed.mean()})
    out = tmp[["src_ip", "dst_ip", "_w"]].join(src, on=["src_ip", "_w"]).join(dst, on=["dst_ip", "_w"])
    out = out.drop(columns=["src_ip", "dst_ip", "_w"])
    for c in ("src_n_flows", "src_n_dst", "src_n_dport", "src_bytes_out",
              "dst_n_flows", "dst_n_src", "dst_n_dport"):
        out[c] = np.log1p(out[c].clip(lower=0))
    return out.replace([np.inf, -np.inf], np.nan).fillna(0).astype(np.float32)


class Familiarity:
    """Kompaktowa znajomość celu/pary/usługi przez stałowymiarowe liczniki haszowane."""
    def __init__(self, n_buckets=32768):
        self.n_buckets = n_buckets

    def _bucket(self, values):
        h = pd.util.hash_pandas_object(values.astype(str), index=False).to_numpy(dtype=np.uint64)
        return (h % np.uint64(self.n_buckets)).astype(np.int64)

    def fit(self, df):
        self.dst = np.bincount(self._bucket(df.dst_ip), minlength=self.n_buckets).astype(np.float32)
        self.pair = np.bincount(self._bucket(df.src_ip.astype(str) + ">" + df.dst_ip.astype(str)),
                                minlength=self.n_buckets).astype(np.float32)
        self.svc = np.bincount(self._bucket(df.dst_ip.astype(str) + ":" + df.dst_port.astype(str)),
                               minlength=self.n_buckets).astype(np.float32)
        return self

    def transform(self, df):
        out = pd.DataFrame(index=df.index)
        out["fam_dst"] = np.log1p(self.dst[self._bucket(df.dst_ip)])
        out["fam_pair"] = np.log1p(self.pair[self._bucket(df.src_ip.astype(str) + ">" + df.dst_ip.astype(str))])
        out["fam_svc"] = np.log1p(self.svc[self._bucket(df.dst_ip.astype(str) + ":" + df.dst_port.astype(str))])
        return out.astype(np.float32)

In [58]:
def regularity_features(df, min_flows=5):
    """Regularność inter-arrivalów per (źródło, cel, port, doba), przypisana do przepływów."""
    out = pd.DataFrame(index=df.index, data={"regularity": np.zeros(len(df), dtype=np.float32)})
    if len(df) == 0:
        return out
    tmp = pd.DataFrame({"_pos": np.arange(len(df)), "src_ip": df.src_ip.to_numpy(),
                        "dst_ip": df.dst_ip.to_numpy(), "dst_port": df.dst_port.to_numpy(),
                        "_day": (pd.to_numeric(df.ts, errors="coerce").fillna(0).to_numpy() // 86400),
                        "_ts": pd.to_numeric(df.ts, errors="coerce").fillna(0).to_numpy()})
    keys = ["src_ip", "dst_ip", "dst_port", "_day"]
    tmp = tmp.sort_values(keys + ["_ts"], kind="stable")
    grouped = tmp.groupby(keys, sort=False)
    count = grouped["_ts"].transform("size")
    tmp["_gap"] = grouped["_ts"].diff()
    gaps = tmp.groupby(keys, sort=False)["_gap"]
    mean_gap = gaps.transform("mean").fillna(0)
    std_gap = gaps.transform("std").fillna(0)
    cv = std_gap / (mean_gap.abs() + 1e-6)
    tmp["_regularity"] = (1.0 / (1.0 + cv)).where(count >= min_flows, 0.0)
    values = np.zeros(len(df), dtype=np.float32)
    values[tmp["_pos"].to_numpy()] = tmp["_regularity"].to_numpy(dtype=np.float32)
    out["regularity"] = values
    return out


def burst_features(df, window=60):
    """Tempo po usłudze, udział krótkich połączeń oraz entropia portów źródła w oknie."""
    cols = ["src_service_n", "src_service_short_ratio", "src_port_entropy"]
    if len(df) == 0:
        return pd.DataFrame(index=df.index, columns=cols, dtype=np.float32)
    tmp = df[["src_ip", "dst_ip", "dst_port", "duration"]].copy()
    tmp["_w"] = (pd.to_numeric(df.ts, errors="coerce").fillna(0) // window).astype(np.int64)
    tmp["_duration"] = pd.to_numeric(tmp.duration, errors="coerce").fillna(0).clip(lower=0)
    service_keys = ["src_ip", "dst_ip", "dst_port", "_w"]
    service = tmp.groupby(service_keys, sort=False).agg(
        _n=("_duration", "size"), _short=("_duration", lambda x: (x <= 1.0).mean()))
    out = tmp[service_keys].join(service, on=service_keys)
    out["src_service_n"] = np.log1p(out["_n"])
    out["src_service_short_ratio"] = out["_short"]
    port_counts = tmp.groupby(["src_ip", "_w", "dst_port"], sort=False).size().rename("_count").reset_index()
    totals = port_counts.groupby(["src_ip", "_w"], sort=False)["_count"].transform("sum")
    proportions = port_counts["_count"] / totals
    port_counts["_term"] = -proportions * np.log(proportions.clip(lower=1e-12))
    entropy = port_counts.groupby(["src_ip", "_w"], sort=False)["_term"].sum().rename("src_port_entropy")
    out = out.join(entropy, on=["src_ip", "_w"])
    return out[cols].fillna(0).astype(np.float32)

## Detektor

In [59]:
class Detector:
    """Detektor MiniBatchKMeans z progową decyzją i API score/predict."""
    def __init__(self, n_clusters=32, sample=60_000, seed=42, target_fph=2.5):
        self.n_clusters = n_clusters
        self.sample = sample
        self.seed = seed
        self.target_fph = target_fph

    def _X(self, df):
        return (flow_features(df)
                .join(window_features(df))
                .join(self.fam.transform(df))
                .join(regularity_features(df))
                .join(burst_features(df))
                .to_numpy(dtype=np.float32))

    @staticmethod
    def _hours(df):
        if len(df) < 2:
            return 1.0 / 60
        span = float(pd.to_numeric(df.ts, errors="coerce").max() -
                     pd.to_numeric(df.ts, errors="coerce").min()) / 3600
        return max(span, 1.0 / 60)

    def _score_X(self, X):
        if len(X) == 0:
            return np.empty(0, dtype=np.float32)
        z = self.scaler.transform(X)
        return pairwise_distances_argmin_min(z, self.model.cluster_centers_)[1].astype(np.float32)

    def fit(self, df):
        if len(df) < 10:
            raise ValueError("Do dopasowania potrzeba co najmniej 10 normalnych przepływów.")
        ordered = df.sort_values("ts", kind="stable")
        days = (pd.to_numeric(ordered.ts, errors="coerce") // 86400).to_numpy()
        unique_days = np.unique(days)
        if len(unique_days) >= 2:
            n_cal_days = max(1, int(np.ceil(0.2 * len(unique_days))))
            first_cal_day = unique_days[-n_cal_days]
            fit_df, cal_df = ordered.loc[days < first_cal_day], ordered.loc[days >= first_cal_day]
        else:
            cut = max(1, min(len(ordered) - 1, int(0.8 * len(ordered))))
            fit_df, cal_df = ordered.iloc[:cut], ordered.iloc[cut:]
        if len(fit_df) < 5 or len(cal_df) < 2:
            cut = max(1, min(len(ordered) - 1, int(0.8 * len(ordered))))
            fit_df, cal_df = ordered.iloc[:cut], ordered.iloc[cut:]

        self.fam = Familiarity().fit(fit_df)
        X_fit = self._X(fit_df)
        rng = np.random.default_rng(self.seed)
        if len(X_fit) > self.sample:
            X_fit = X_fit[rng.choice(len(X_fit), self.sample, replace=False)]
        self.scaler = StandardScaler().fit(X_fit)
        z_fit = self.scaler.transform(X_fit)
        self.model = MiniBatchKMeans(n_clusters=min(self.n_clusters, len(z_fit)),
                                     batch_size=4096, n_init=3,
                                     random_state=self.seed).fit(z_fit)

        cal_scores = self._score_X(self._X(cal_df))
        cal_hours = self._hours(cal_df)
        allowed = min(max(int(np.floor(self.target_fph * cal_hours)), 1), max(len(cal_scores) - 1, 0))
        threshold_index = max(0, len(cal_scores) - allowed - 1)
        self.threshold = float(np.partition(cal_scores, threshold_index)[threshold_index])
        self.calibration_hours = cal_hours
        self.calibration_fph = float(np.sum(cal_scores > self.threshold) / cal_hours)
        self.max_fp_per_hour = 5.0
        self.feature_count = X_fit.shape[1]
        return self

    def score(self, df):
        return self._score_X(self._X(df))

    def predict(self, df):
        return (self.score(df) > self.threshold).astype(np.int8)

## Trening

In [60]:
det = Detector().fit(train)


## Zapis modelu

In [61]:
with open("model.pkl", "wb") as f:
    cloudpickle.dump(det, f)

# Kontrola artefaktu po ponownym wczytaniu.
with open("model.pkl", "rb") as f:
    loaded = cloudpickle.load(f)
check = train.head(1000)
assert np.allclose(loaded.score(check), det.score(check))
assert np.array_equal(loaded.predict(check), det.predict(check))
print(f"zapisano model.pkl ({os.path.getsize('model.pkl') / (1024 ** 2):.2f} MiB)")

zapisano model.pkl (0.63 MiB)


## Ewaluacja
Plik `test.csv.gz` z etykietami w tym samym folderze.

In [66]:
def evaluate(model, path="test.csv.gz"):
    """Etykiety testowe służą wyłącznie do obliczenia raportu ewaluacyjnego."""
    test = pd.read_csv(path, keep_default_na=False)
    X = test.drop(columns=[c for c in ("label", "phase", "raw_label") if c in test.columns])

    start = time.perf_counter()
    scores = np.asarray(model.score(X))
    alarm = scores > model.threshold
    elapsed = time.perf_counter() - start
    y = test.label.to_numpy().astype(bool)

    hours = max(model._hours(test), 1 / 60)
    false_alarms = int((alarm & ~y).sum())
    fpr = false_alarms / max(int((~y).sum()), 1)
    detected = int((alarm & y).sum())
    metrics = {
        "flows": len(test),
        "false_alarms": false_alarms,
        "fph": false_alarms / hours,
        "true_positives": detected,
        "false_negatives": int(y.sum()) - detected,
        "recall": detected / max(int(y.sum()), 1),
        "precision": detected / max(int(alarm.sum()), 1),
        "elapsed_seconds": elapsed,
        "flows_per_second": len(test) / max(elapsed, 1e-9),
        "attack_flows": int(y.sum()),
        "alarm_flows": int(alarm.sum()),
    }
    print(f"przepływy: {metrics['flows']:,} (atak: {metrics['attack_flows']:,}), sieci: {test.network.nunique()}")
    print(f"czułość: {metrics['recall']:.3f}, precyzja: {metrics['precision']:.3f}")
    print(f"fałszywe alarmy: {false_alarms:,} ({metrics['fph']:.2f}/h)")
    print(f"czas score z cechami: {elapsed:.2f} s, {metrics['flows_per_second']:,.0f} przepływów/s")

    phases = []
    for (network, phase), group in test.loc[y].groupby(["network", "phase"]):
        idx = group.index.to_numpy()
        sensitivity = float(alarm[idx].mean())
        phases.append({"sieć": network, "faza": phase, "przepływów": len(group),
                       "alarmów": int(alarm[idx].sum()), "czułość": round(sensitivity, 3),
                       "wykryta": bool(alarm[idx].any() and sensitivity >= 3 * fpr)})
    phase_table = pd.DataFrame(phases)
    if phase_table.empty:
        print("Brak oznaczonych faz ataku w teście.")
        metrics.update({"phase_types_total": 0, "phase_types_detected": 0,
                        "incidents_total": 0, "incidents_detected": 0})
        return phase_table, metrics

    summary = phase_table.groupby("faza").wykryta.agg(["sum", "count"]).rename(
        columns={"sum": "wykryte", "count": "w sieciach"})
    metrics.update({
        "phase_types_total": len(summary),
        "phase_types_detected": int(summary.wykryte.gt(0).sum()),
        "incidents_total": len(phase_table),
        "incidents_detected": int(phase_table.wykryta.sum()),
    })
    print(f"wykryte incydenty (sieć × faza): {metrics['incidents_detected']} z {metrics['incidents_total']}")
    print(f"wykryte typy faz: {metrics['phase_types_detected']} z {metrics['phase_types_total']}")
    print(summary)
    return phase_table, metrics

In [67]:
phase_results, evaluation_metrics = evaluate(det)


przepływy: 515,018 (atak: 290,451), sieci: 5
czułość: 0.775, precyzja: 0.995
fałszywe alarmy: 1,203 (50.13/h)
czas score z cechami: 21.27 s, 24,216 przepływów/s
wykryte incydenty (sieć × faza): 19 z 30
wykryte typy faz: 5 z 7
                  wykryte  w sieciach
faza                                 
dns_brute_force         5           5
dns_exfiltration        0           1
host_discovery          5           5
online_cracking         2           4
service_scan            5           5
web_scan                2           5
webshell                0           5


## Zastosowane rozwiązanie: MiniBatchKMeans

Model używa standaryzowanych cech przepływu, agregatów 60-sekundowych, liczników znajomości celu oraz cech burstów i regularności. `MiniBatchKMeans` dopasowuje 32 centroidy na próbce do 60 000 przepływów z chronologicznie wcześniejszej części normalnego `train.csv.gz`. Skaler i model nie widzą danych testowych ani ich etykiet.

`score(df)` zwraca odległość od najbliższego centroidu; im wyższa, tym bardziej nietypowy przepływ. `predict(df)` porównuje wynik z progiem kalibrowanym na późniejszym fragmencie normalnego treningu. Kalibracja ustawia nominalnie 2.5 alarmu/h; test pokazuje, jak model generalizuje.

Ewaluacja na 515 018 przepływach testowych: czułość **0.775**, precyzja **0.995**, **5 z 7** typów faz i **19 z 30** incydentów wykrytych według kryterium w ewaluatorze. `score()` wraz z obliczeniem cech trwał **21.61 s** (około 23.8 tys. przepływów/s); zapisany `model.pkl` ma **0.63 MiB**.

Agregaty cech są liczone na całej przekazanej ramce, więc model działa w trybie batch. Przy uruchomieniu `score(df)`/`predict(df)` przekazuj wejściowe kolumny przepływu bez `label`, `phase` i `raw_label`.

In [68]:
# Końcowe porównanie trzech kryteriów; próg pochodzi wyłącznie z normalnego train.
model_size_mib = os.path.getsize("model.pkl") / (1024 ** 2)
phase_result = (
    f"nieosiągnięte przy limicie; {evaluation_metrics['phase_types_detected']}/"
    f"{evaluation_metrics['phase_types_total']} przy {evaluation_metrics['fph']:.2f} FP/h"
)
phase_status = "spełnia limit" if evaluation_metrics["fph"] <= 5 else "limit 5 FP/h przekroczony"

final_results = pd.DataFrame([
    {"Kryterium": "Rozmiar modelu", "Wynik": f"{model_size_mib:.2f} MiB",
     "Ocena": "model.pkl"},
    {"Kryterium": "Szybkość predykcji", "Wynik": f"{evaluation_metrics['flows_per_second']:,.0f} przepływów/s",
     "Ocena": f"{evaluation_metrics['elapsed_seconds']:.2f} s, razem z cechami"},
    {"Kryterium": "Wykryte typy faz przy limicie ≤5 FP/h", "Wynik": phase_result,
     "Ocena": phase_status},
])
final_results

,Kryterium,Wynik,Ocena
0,Rozmiar modelu,0.63 MiB,model.pkl
1,Szybkość predykcji,"24,216 przepływów/s","21.27 s, razem z cechami"
2,Wykryte typy faz przy limicie ≤5 FP/h,nieosiągnięte przy limicie; 5/7 przy 50.13 FP/h,limit 5 FP/h przekroczony


### Szczegółowe wyniki klasyfikacji

Tabela pokazuje jakość alarmowania oraz ich liczbę. „Alarmy trafione” to przepływy alarmowe z etykietą ataku; „ataki wykryte” to liczba wykrytych przepływów ataku (TP).

In [70]:
# Pokazujemy wskaźniki jakości oraz liczbę wszystkich alarmów i trafień.
classification_results = pd.DataFrame([
    {"Wskaźnik": "Czułość", "Wynik": f"{evaluation_metrics['recall']:.1%}"},
    {"Wskaźnik": "Precyzja", "Wynik": f"{evaluation_metrics['precision']:.1%}"},
    {"Wskaźnik": "Alarmy ogółem", "Wynik": f"{evaluation_metrics['alarm_flows']:,}"},
    {"Wskaźnik": "Alarmy trafione (TP)", "Wynik": f"{evaluation_metrics['true_positives']:,}"},
    {"Wskaźnik": "Ataki w teście", "Wynik": f"{evaluation_metrics['attack_flows']:,}"},
    {"Wskaźnik": "Fałszywe alarmy (FP)", "Wynik": f"{evaluation_metrics['false_alarms']:,}"},
    {"Wskaźnik": "Ataki niewykryte (FN)", "Wynik": f"{evaluation_metrics['false_negatives']:,}"},
])
classification_results

,Wskaźnik,Wynik
0,Czułość,77.5%
1,Precyzja,99.5%
2,Alarmy ogółem,"226,337"
3,Alarmy trafione (TP),"225,134"
4,Ataki w teście,"290,451"
5,Fałszywe alarmy (FP),"1,203"
6,Ataki niewykryte (FN),"65,317"
